##THIS NOTEBOOK FILE IS FOR RUNNING MODEL ON COLLAB AND EXPOSING LIKE API . 

Because running local model for spec contraint machine is headace . this notebook solve that headhace

In [1]:
# Install dependencies and Ollama
# Install dependencies and Ollama
!apt-get update && apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh







Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 https://cli.github.com/packages stable InRelease [3,917 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,296 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [

## Step 3 — Pull a model
Choose based on your GPU:
- **T4 (15 GB VRAM):** `qwen2.5-coder:7b`, `gemma3:12b`
- **A100 (40 GB VRAM):** `gemma3:27b`, `qwen2.5:32b`

In [10]:
import subprocess, time, os

# 1. Kill any existing Ollama processes to start fresh
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)

# 2. Set environment variables
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_ORIGINS"] = "*"

# 3. Start Ollama in the background and redirect output to a log file
with open("ollama.log", "w") as log_file:
    subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=log_file,
        stderr=log_file
    )

# 4. Wait for the server to be ready
print("⏳ Starting Ollama...")
for _ in range(10):
    try:
        import requests
        resp = requests.get("http://localhost:11434/")
        if resp.status_code == 200:
            print("✅ Ollama is up and running!")
            break
    except:
        time.sleep(2)
else:
    print("❌ Ollama failed to start. Check 'ollama.log' for details.")

⏳ Starting Ollama...
✅ Ollama is up and running!


In [8]:
# Change MODEL_NAME to whatever you want to use
# MODEL_NAME = "qwen2.5-coder:7b"
MODEL_NAME = "llama3.1:8b"

!ollama pull {MODEL_NAME}
print(f"✅ Model {MODEL_NAME} ready!")


✅ Model llama3.1:8b ready!


In [ ]:

!pip install pyngrok -q
from pyngrok import ngrok

# Paste your token here from ngrok for exposing you model like api
authtoken=""
ngrok.set_auth_token(authtoken)

# Kill any existing tunnels
ngrok.kill()

# Open tunnel on Ollama's port
tunnel = ngrok.connect(11434, "http")

# Correctly extract the URL string
public_url = tunnel.public_url

print(f"✅ Public API URL: {public_url}")
print(f"\n📌 Use this as your base URL in any OpenAI-compatible client:")
print(f"   {public_url}/api/generate       ← Ollama native")
print(f"   {public_url}/v1/chat/completions ← OpenAI-compatible (Ollama ≥0.1.24)")

✅ Public API URL: https://hummingly-athetoid-wanita.ngrok-free.dev

📌 Use this as your base URL in any OpenAI-compatible client:
   https://hummingly-athetoid-wanita.ngrok-free.dev/api/generate       ← Ollama native
   https://hummingly-athetoid-wanita.ngrok-free.dev/v1/chat/completions ← OpenAI-compatible (Ollama ≥0.1.24)


#step 6 - TEST the public api

In [16]:
# Test via the public URL
import requests

# Ensure base is clean
base = public_url.rstrip("/")

try:
    # Increased timeout to 120 seconds to allow for model loading/processing
    resp = requests.post(
        f"{base}/api/generate",
        json={"model": MODEL_NAME, "prompt": "What is 2+2?", "stream": False},
        timeout=120
    )

    print("Status:", resp.status_code)
    if resp.status_code == 200:
        print("Answer:", resp.json().get("response", ""))
    else:
        print("Error:", resp.text)
except requests.exceptions.ReadTimeout:
    print("❌ Error: The request timed out. The model might still be loading or the server is busy.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")

https://hummingly-athetoid-wanita.ngrok-free.dev


## Step 7 — OpenAI-compatible chat endpoint
Ollama ≥ 0.1.24 exposes `/v1/chat/completions` — works with the `openai` Python SDK.

In [13]:
# Install OpenAI SDK if needed
!pip install openai -q

from openai import OpenAI

client = OpenAI(
    base_url=f"{public_url}/v1",
    api_key="ollama"  # required by SDK but not checked by Ollama
)

completion = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Explain what an API is in 2 sentences."}]
)
print(completion.choices[0].message.content)

API stands for Application Programming Interface, which is a set of rules and protocols that software applications use to communicate with each other. It allows different software systems to interact with each other by exchanging data in a structured format.


### Debugging & Health Check
Run this cell to verify if Ollama is actually listening and if ngrok is active.

In [21]:
import requests

print("--- Final Connection Check ---")

# 1. Check local health
try:
    local = requests.get("http://localhost:11434/").status_code
    print(f"✅ Local Ollama: Running (Status {local})")
except Exception as e:
    print(f"❌ Local Ollama: Failed ({e})")

# 2. Check public tunnel health
try:
    # Using the public_url variable from your ngrok cell
    public_health = requests.get(f"{public_url}/").status_code
    print(f"✅ Public Tunnel: Connected! ({public_url})")
except Exception as e:
    print(f"❌ Public Tunnel: Failed to reach Ollama via ngrok ({e})")

# 3. List models via public URL
try:
    models = requests.get(f"{public_url}/api/tags").json()
    model_names = [m['name'] for m in models.get('models', [])]
    print(f"✅ Reachable Models: {model_names}")
except:
    print("❌ Models: Could not list models via public URL.")

--- Diagnostic Report ---
❌ Ollama Local: Not reachable (HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7ab10eccfb00>: Failed to establish a new connection: [Errno 111] Connection refused')))
✅ Ngrok Active: ['https://hummingly-athetoid-wanita.ngrok-free.dev']
❌ Models: Could not retrieve list.


In [ ]:
%xterm

In [ ]:
!pip install colab-xterm
%load_ext colabxterm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 5.9 MB/s eta 0:00:00


In [11]:
# Prevents Colab from disconnecting due to idle
import time
print("Keeping session alive... Press Stop to end.")
while True:
    time.sleep(60)
    print("⏱️ Still running...")

Keeping session alive... Press Stop to end.
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running...
⏱️ Still running

KeyboardInterrupt: 